In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import psutil
import plotly.graph_objects as go
import chart_studio
import chart_studio.plotly as py

import plotly.io as pio

chart_studio.tools.set_config_file(world_readable=True,
                             sharing='public')


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
stonks = pd.read_csv("stonks.csv")

In [ ]:
stonks = stonks.iloc[::-1]
stonks.head()

In [ ]:
split_percent = 0.7
val_percent = 0.1

close_data = stonks[["open", "high", "low", "close", "close"]].values.reshape((-1,5))
scaler = MinMaxScaler()
scaler.fit(close_data)

close_data = scaler.transform(close_data)
split = int(split_percent*len(close_data))
val = int((val_percent + split_percent) * len(close_data))
close_train = close_data[:split]
close_val = close_data[split:val]
close_test = close_data[val:]

date_train = stonks.timestamp[:split]
date_val = stonks.timestamp[split:val]
date_test = stonks.timestamp[val:]

close_train[0:, 4:]

In [ ]:
look_back = 20
num_epochs = 92

train_generator = keras.preprocessing.sequence.TimeseriesGenerator(close_train, close_train, length=look_back, batch_size=20)   
val_generator = keras.preprocessing.sequence.TimeseriesGenerator(close_val, close_val, length=look_back, batch_size=20)     
test_generator = keras.preprocessing.sequence.TimeseriesGenerator(close_test, close_test, length=look_back, batch_size=1)

model = keras.Sequential([
    keras.layers.LSTM(256,
                      activation="relu",
                      #recurrent_activation="sigmoid",
                      use_bias=True,
                      input_shape=(look_back, 5)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(5),
])
model.compile(optimizer="adam", loss="mse", metrics=["accuracy"])


model.fit(train_generator, epochs=num_epochs, verbose=1, validation_data=val_generator)

In [ ]:
prediction = model.predict(test_generator)

def denorm(d): 
    return scaler.inverse_transform(d)[0:, 4:].reshape((-1))
close_train_1 = denorm(close_train)
close_val_1 = denorm(close_val)
close_test_1 = denorm(close_test)
prediction_1 = denorm(prediction)

diff = close_test_1[look_back:] - prediction_1
avg_diff = np.mean(diff)
print(avg_diff)

def predict(num_prediction, before_lookback, model):
    if (before_lookback > 0):
        prediction_list = close_data[-look_back - before_lookback:-before_lookback]
    else:
        prediction_list = close_data[-look_back:]
    for _ in range(num_prediction):
        x = prediction_list[-look_back:]
#         print("Pred list", prediction_list)
        x = x.reshape((1, look_back, 5))
        out = model.predict(x)
#         print("Out", out)
        prediction_list = np.append(prediction_list, out, axis=0)
    prediction_list = prediction_list[look_back:]
        
    return prediction_list
    
def predict_dates(num_prediction, before_lookback):
    last_date = stonks.timestamp.values[-1 - before_lookback]
    prediction_dates = pd.date_range(last_date, periods=num_prediction+1+before_lookback).tolist()
    return prediction_dates

num_prediction = 4
before_lookback = 0
forecast = denorm(predict(num_prediction, before_lookback, model))
forecast_dates = predict_dates(num_prediction, before_lookback)
prediction_1 = np.concatenate([[None] * look_back, prediction_1 + avg_diff])
trace1 = go.Scatter(
    x = np.concatenate([date_train, date_val]),
    y = np.concatenate([close_train_1, close_val_1]),
    mode = 'lines',
    name = 'Real Data - trained'
)
trace2 = go.Scatter(
    x = date_test,
    y = prediction_1,
    mode = 'lines',
    name = 'Prediction'
)
trace3 = go.Scatter(
    x = date_test,
    y = close_test_1,
    mode='lines',
    name = 'Real Data - untrained'
)
forecast = go.Scatter(
    x = forecast_dates,
    y = forecast + avg_diff,
    mode = 'lines+markers+text',
    texttemplate = "%{y:.2f}",
    name = 'Forecast'
)

layout = go.Layout(
    title = "SPY",
    xaxis = {'title' : "Date"},
    yaxis = {'title' : "Close"}
)
fig = go.Figure(data=[trace1, trace2, trace3, forecast], layout=layout)
fig.show(renderer="browser")
# py.plot(fig, filename = 'basic-line', renderer="browser", auto_open=True)

In [ ]:
pio.write_html(fig, file='tracing_pred.html', auto_open=True)


In [ ]:
model.save('good_model_bak_2.h5')

In [ ]:
model = keras.models.load_model("good_model_bak_2.h5", custom_objects={'loss': customloss()})